In [1]:
# Generelle moduler og funksjonsbeskrivelser brukt i forelesningen
from numpy import sin, cos, pi, exp, angle, abs, unwrap, log10
import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [8.00, 4.5]
plt.rcParams["figure.autolayout"] = True
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.xmargin"] = 0.0

%matplotlib inline

<img src="figurer/NTNU_logo_vertical.svg" align="left" width="30%">
<br clear="all" />
<br></br>

# Intro til tidsdiskrete system

* **Emne: AIS2102 - Dynamiske system**
* ***dato***: 22. januar 2025
* **Foreleser: Kai Erik Hoff**

# Hvorfor diskretisering?
<br>
<br>
<img src="figurer/disc_loop_1.png" style="width: 80%; margin-left: 10%" />

* *Hvordan kan vi programmere en digital enhet til å gjengi egenskapene gitt av en differensiallikning på en god måte?*

# Hva skal vi lære?

1. Hvordan "oversette" et LTI-system beskrevet med f.eks. en transferfunksjon $G(s)$ til et format som enkelt kan implementeres på en digital enhet (mikrokontroller, datamaskin etc.)

2. Hvordan verifisere matematisk at reguleringssløyfen fortsatt tilfredsstiller ønskede krav når man tar høyde for implementasjon på digital enhet.

3. Grunnleggende om Digitale Filtre
    * Det eneste som i realiteten skiller en regulator fra et filter er hvordan det anvendes
        * *De to fagområdene har dessverre en tendens til å benytte ulike notasjoner, som kan få det til å virke som om forskjellen er større enn den er*.
    * Matematikken bak digitale regulatorer og digitale filtre er den samme

# Hva menes med diskretisering?

### Tidsdiskrete signal
* Fagbegrep som kan anses som ensbetydende med "måleserie"
* Signalet er kun definert i de eksakte tidspunktene hvor det er tatt en måling

<img src="figurer/discretized_signal.svg" style="width: 90%; margin-left: 5%" />

### Tidsdiskrete system
* System som er laget for å håntere et tidsdiskret inngangssignal, og produsere et tidsdiskret ugangssignal
* Eksisterer typisk kun i form av programvare, femfor f.eks. en elektrisk filterkrets.

<div style="height=400px">
<img src="figurer/rc_circuit.svg" style="float:left; width: 40%"/>
</div>

```c++
class LowpassFilter {
  private:             
    float alpha;        // discrete filter coefficient
    float prev_y;       // filter state variable
  public:
    LowpassFilter(float R, float C, float T_s){
      this->alpha = exp(-R*C*T_s)
      this->prev_y = 0.0
    float process_sample(float x){
      float y = alpha*this_>prev_y + (1-alpha)*x
      this->prev_y = y
      return y
    }
};

```

* Diskretisering omhandler konverteringen av en systemmodell fra tidskontinuerlig domene til tidsdiskret domene
* Har dere programmert en PID kontroller, har dere allerede utført en form for diskretisering

# Tre mulige formål med diskretisering:

1. Diskretisere regulatoren $G_c(s)$ slik at den kan programmeres på f.eks. en mikrokontroller.

2. Diskretisere prosessen $G_p(s)$ for å kunne analysere regulatorsløyfen etter diskretisering.

3. For å kunne simulere regulatorsløyfen (program som f.eks. Simulink gjør diskretisering for oss).

# Arbeider og pensumlitteratur

* Arbeider
    - To obligatoriske øvinger 
    - Én laboppgave med rapport (inngår i mappen)
        * Frist i februar (dato kommer)
        * Utføres i grupper på 2

* Faglitteratur
    - [Discrete-time signals and systems](https://techteach.no/publications/discretetime_signals_systems/discrete.pdf)<br>av Finn Haugen
    - (Kapittel 13, Control Systems Engineering, Norman Nise)<br>Relevant men ikke påkrevd
    - [Videoserie om Diskrét Reguleringsteknikk av Brian Douglas](https://www.youtube.com/playlist?list=PLUMWjy5jgHK0MLv6Ksf-NHi7Ur8NRNU4Z)

# Arbeidsflyt digital regulator (Arduino)


> ```c++
> int inputValue = analogRead(INPUT_PIN); // A/D conversion
> int outputValue = doControllerStuff(inputValue); // Evaluate output of controller
> analogWrite(OUTPUT_PIN, outputValue); // D/A conversion
> ```

* Hver operasjon utføres sekvensielt, ved diskréte tidspunkt.

* Hver av operasjonene får sin egen systemblokk i en systemmodell.

<img src="figurer/disc_reg_1.png" style="width: 60%; margin-left: 20%" />

# Hybridsystem

<img src="figurer/hybrid_sys_1.png" style="width: 90%; margin-left: 5%" />

* Sammensetning av *tidskontinuerlige* system ($G_p(s)$) og *tidsdiskrete* system ($G_c(z)$)

# Fokus for diskretiseringsmodul
<img src="figurer/disc_reg_2.png" style="width: 60%; margin-left: 20%" />

* Oppgave: sørge for at det digitale LTI-systemet oppnår *tilnærmet* samme egenskaper som $G(s)$

* Vi begynner med diskretisering av inngangs-signal (A/D omforming)